# Embedding Evaluation

In [ ]:
import numpy as np
import sys
import os
import pandas as pd
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

from pathlib import Path
sys.path.append(os.path.abspath('..'))

from src.skipgram import SkipGramModel,train_model
from src.negative_sampling import build_unigram_table
from src.tokenizer import Tokenizer



In [ ]:



with open("../data/corpus.txt", "r", encoding='utf-8') as f:
    raw_data = f.read()
    # Split the string by spaces and convert each "number" to an actual integer
    corpus_ids = [int(id_str) for id_str in raw_data.split()]
    
    
#tok.build_vocab(raw_data)


print(f"First 10 IDs: {corpus_ids[:10]}")

In [ ]:
df = pd.read_csv('./../data/IMDB Dataset.csv')


#print(df.head())


tok = Tokenizer()

reviews = df['review'].tolist()

# for review in df['review']:
#     reviews.append(review.replace('\n',''))
    
    
# print(reviews[:20])

tok.build_vocab(reviews)

In [ ]:
unigram_table = build_unigram_table(corpus_ids,10000)

### Training

In [ ]:
model = SkipGramModel(10000,100)

train_model(model,[corpus_ids[:200000]],unigram_table,10,3,0.025)

In [ ]:
def find_nearest_neighbors(query_word,word2idx,idx2word,normalized_matrix,top_k=5):
    if query_word not in word2idx:
        print(f"{query_word} is not in the dictionary")
        return
    
    
    #1 get the id and vector for the query word
    query_id = word2idx[query_word]
    query_vector = normalized_matrix[query_id]
    
    #2. cosine similarity 
    similarities = np.dot(normalized_matrix,query_vector)
    
    #3. sort  the indices based on similarity scores
    sorted_indices = np.argsort(similarities)
    
    #4. Grab the top k indices
    #remove the last index and reverse,
    best_indices = sorted_indices[-(top_k + 1):-1][::-1]
    
    print(f"words most similar to '{query_word}':")
    for idx in best_indices:
        print(f" {idx2word[idx]} (Score: {similarities[idx]:.4f})")
            

In [ ]:
norms = np.linalg.norm(model.W1, axis=1,keepdims=True)

norms[norms == 0] = 1e-10

normalized_W1 = model.W1 / norms

idx2word = {idx: word for word, idx in tok.word_to_id.items()}

print(norms.shape)

In [ ]:
find_nearest_neighbors("scary",tok.word_to_id,idx2word,normalized_W1)

In [ ]:
def test_analogy(word_a,word_b,word_c,word2idx,idx2word,matrix,top_k=10):
    #Make sure all words are in our dictionary
    for w in [word_a,word_b,word_c]:
        if w not in word2idx:
            print(f"'{w}' is not in vocabulary")
            return
        
    #1. Get the vectors
    vec_a = matrix[word2idx[word_a]]
    vec_b = matrix[word2idx[word_b]]
    vec_c = matrix[word2idx[word_c]]
    print(matrix.shape)
    print(vec_a.shape)
    
    print(word2idx[word_a])
    
    
    #2. Do the math: target = A - B + C
    target_vector = vec_a - vec_b + vec_c
    
    #3. Calculate similarities across the whole matrix
    similarities = np.dot(matrix, target_vector)
    
    #4. Sort and slice the top K (including the #1 best match)
    sorted_indices =  np.argsort(similarities)
    best_indices = sorted_indices[-top_k:][::-1]
    
    
    print(f"Analogy: {word_a} - {word_b} + {word_c}")
    
    for idx in best_indices:
        if idx2word[idx] == word_a or idx2word[idx] == word_b or idx2word[idx] == word_c:
            continue
        print(f" {idx2word[idx]} (Score: {similarities[idx]:.4f})")

In [ ]:
test_analogy("king", "man", "woman", tok.word_to_id, idx2word, normalized_W1)

## VISUALIZATION

In [ ]:
#1. set up the calculator to give us a 2 dimension
tsne_model = TSNE(n_components=2, random_state=42)



#2. Feed it our 300 vectors to get the 2D versions back
vectors_to_plot = normalized_W1[:300]

labels = [idx2word[i] for i in range(300)]

embeddings_2d = tsne_model.fit_transform(vectors_to_plot)



In [ ]:
#Extract the X and Y columns 
x_coords = embeddings_2d[:,0]
y_coords = embeddings_2d[:,1]


plt.figure(figsize=(12,10))
plt.scatter(x_coords,y_coords, color="blue")


for i in range(300):
    plt.annotate(labels[i],(x_coords[i],y_coords[i]))
    
plt.show()